# Review Analytics

## Importing Dependencies

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
pd.set_option('display.max_columns', 100) # Show all columns when looking at dataframe
import warnings
warnings.filterwarnings('ignore') # remove warnings from the notebook
import matplotlib
import nltk

## Importing Data

In [0]:
df= pd.read_csv("amazonreviews.tsv", sep='\t')

In [21]:
df.head()

,label,review
0,pos,Stuning even for the non-gamer: This sound tra...
1,pos,The best soundtrack ever to anything.: I'm rea...
2,pos,Amazing!: This soundtrack is my favorite music...
3,pos,Excellent Soundtrack: I truly like this soundt...
4,pos,"Remember, Pull Your Jaw Off The Floor After He..."


In [22]:
# Check for the existence of NaN values in a cell:
df.isnull().sum()

label     0
review    0
dtype: int64

In [23]:
# Detect & remove empty strings
blanks = []  # start with an empty list

for i,lb,rv in df.itertuples():  # iterate over the DataFrame
    if type(rv)==str:            # avoid NaN values
        if rv.isspace():         # test 'review' for whitespace
            blanks.append(i)     # add matching index numbers to the list
        
print(len(blanks), 'blanks: ', blanks)

0 blanks:  []


In [0]:
# Wordcount in each Reviews
df['Word_Count'] = [len(review.split()) for review in df['review']]

In [0]:
df['Chracter_Count']= [len(review) for review in df['review']]

In [0]:
df['upper_case'] = [sum(char.isupper() for char in review) for review in df['review']]

In [0]:
import string
df['punctuation'] = [sum(char in string.punctuation for char in review) for review in df['review']] 

In [28]:
df.head()

,label,review,Word_Count,Chracter_Count,upper_case,punctuation
0,pos,Stuning even for the non-gamer: This sound tra...,80,426,10,11
1,pos,The best soundtrack ever to anything.: I'm rea...,97,509,12,14
2,pos,Amazing!: This soundtrack is my favorite music...,129,760,27,40
3,pos,Excellent Soundtrack: I truly like this soundt...,118,743,59,33
4,pos,"Remember, Pull Your Jaw Off The Floor After He...",87,481,31,22


## Emotional Analysis

In [0]:
emotion_df= pd.read_csv('my_table.csv')

In [30]:
emotion_df.shape

(7528, 43)

In [31]:
emotion_df.columns

Index(['ID', 'CITY', 'COUN', 'SUBJ', 'SEX', 'AGE', 'RELI', 'PRAC', 'FOCC',
       'MOCC', 'FIEL', 'EMOT', 'WHEN', 'LONG', 'INTS', 'ERGO', 'TROPHO',
       'TEMPER', 'EXPRES', 'MOVE', 'EXP1', 'EXP2', 'EXP10', 'PARAL', 'CON',
       'EXPC', 'PLEA', 'PLAN', 'FAIR', 'CAUS', 'COPING', 'MORL', 'SELF',
       'RELA', 'VERBAL', 'NEUTRO', 'Field1', 'Field3', 'Field2', 'MYKEY',
       'SIT', 'STATE', 'Unnamed: 42'],
      dtype='object')

In [0]:
emotion_df= emotion_df[['Field1', 'SIT']]

In [33]:
emotion_df.head()

,Field1,SIT
0,joy,"During the period of being in love, each time ..."
1,fear,When I was involved in a traffic accident.
2,anger,It infuriated me when the motorist did not let...
3,sadness,When I lost the person who meant the most to me.
4,disgust,The time I saw a deer knocked and the sight of...


In [34]:
emotion_df.Field1.value_counts()

joy        1088
guilt      1083
fear       1079
anger      1076
sadness    1074
shame      1066
disgust    1062
Name: Field1, dtype: int64

## Text Preprocessing

In [0]:
import string
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # lower text
    text = text.lower()

    # remove puncutation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize_text
    text = [word for word in word_tokenize(text)]
    
    # removing Stopwords
    text= [w for w in text if not w in stop_words] 

    # remove words that contain numbers
    text = [word for word in text if not any(c.isdigit() for c in word)]
    
    # remove empty tokens
    text = [t for t in text if len(t) > 0]

    # remove words with only one letter
    text = [t for t in text if len(t) > 1]

    # join all
    text = " ".join(text)
    return(text)

In [0]:
emotion_df.SIT= emotion_df.SIT.apply(clean_text)

In [0]:
y = emotion_df.Field1

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y= le.fit_transform(y)

## Vectorization

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(min_df= 2)
X = vectorizer.fit_transform(emotion_df.SIT)

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X.toarray(), y, test_size=0.2, random_state=42)

In [41]:
X_train.shape, y_train.shape

((6022, 4359), (6022,))

In [42]:
X_test.shape, y_test.shape

((1506, 4359), (1506,))

In [43]:
## Fitting Model to the Training Set
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=0, solver='warn', tol=0.0001, verbose=0,
                   warm_start=False)

In [0]:
# Predicting Test Set
y_pred = classifier.predict(X_test)

In [45]:
# Evaluating Results
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
cm = confusion_matrix(y_test, y_pred)
acc= accuracy_score(y_test, y_pred)
print(f"Accuracy : {(acc * 100):.4f} %")

Accuracy : 56.7065 %


In [46]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

       anger       0.70      0.58      0.64       219
     disgust       0.54      0.46      0.50       216
        fear       0.45      0.43      0.44       226
       guilt       0.64      0.70      0.67       234
         joy       0.58      0.69      0.63       191
     sadness       0.57      0.62      0.60       217
       shame       0.48      0.50      0.49       203

    accuracy                           0.57      1506
   macro avg       0.57      0.57      0.57      1506
weighted avg       0.57      0.57      0.57      1506



## Classifing Reviews based on the Above Model

In [0]:
X= df.review.apply(clean_text)
X= vectorizer.transform(X)
df['Emotions'] = classifier.predict(X)

In [0]:
df['Emotions']= list(le.inverse_transform(df['Emotions']))

In [49]:
df.head()

,label,review,Word_Count,Chracter_Count,upper_case,punctuation,Emotions
0,pos,Stuning even for the non-gamer: This sound tra...,80,426,10,11,fear
1,pos,The best soundtrack ever to anything.: I'm rea...,97,509,12,14,disgust
2,pos,Amazing!: This soundtrack is my favorite music...,129,760,27,40,guilt
3,pos,Excellent Soundtrack: I truly like this soundt...,118,743,59,33,guilt
4,pos,"Remember, Pull Your Jaw Off The Floor After He...",87,481,31,22,fear


## Cleaning Reviews

In [0]:
df['Clean_reviews']= df.review.apply(clean_text)

## Most Frequent Unigrams

In [0]:
freq = pd.Series(' '.join(df['Clean_reviews']).split()).value_counts()
freq[:300].to_csv('count.csv')

## Getting Most frequent Bigrams

In [0]:
tokens = nltk.word_tokenize(' '.join(df['Clean_reviews']))
#Create your bigrams
bgs = nltk.bigrams(tokens)
key= []
value= []
fdist = nltk.FreqDist(bgs)
for k,v in fdist.items():
    key.append(k)
    value.append(v)
bi_df= pd.DataFrame({'Bigram': key, 'Count': value})
bi_df= bi_df.sort_values(by= 'Count', ascending= False)
bi_df= bi_df.reset_index(drop= True)
def convertTuple(tup): 
    str =  ' '.join(tup) 
    return str
bi_df.Bigram= bi_df.Bigram.apply(convertTuple)

In [53]:
bi_df.head()

,Bigram,Count
0,read book,419
1,waste money,236
2,would recommend,210
3,one best,183
4,waste time,177


In [0]:
bi_df[:200].to_csv("bi_count.csv", index= False)

## Filtering Based on Sentiment Label

In [0]:
positive= df.loc[df.label== 'pos']
positive= positive.reset_index(drop= True)
negative= df.loc[df.label== 'neg']
negative= negative.reset_index(drop= True)

## Most frequent Unigrams in Positive Reviews

In [0]:
pos_freq = pd.Series(' '.join(positive['Clean_reviews']).split()).value_counts()
pos_freq[:300].to_csv("posi_count.csv")

## Most Frequent Unigrams in Negative Reviews 

In [0]:
neg_freq = pd.Series(' '.join(negative['Clean_reviews']).split()).value_counts()
neg_freq[:300].to_csv("neg_count.csv")

## Most Frequent Bigrams in Positive Reviews

In [0]:
tokens = nltk.word_tokenize(' '.join(positive['Clean_reviews']))
#Create your bigrams
bgs = nltk.bigrams(tokens)
key= []
value= []
fdist = nltk.FreqDist(bgs)
for k,v in fdist.items():
    key.append(k)
    value.append(v)
bi_df_pos= pd.DataFrame({'Bigram': key, 'Count': value})
bi_df_pos= bi_df_pos.sort_values(by= 'Count', ascending= False)
bi_df_pos= bi_df_pos.reset_index(drop= True)
def convertTuple(tup): 
    str =  ' '.join(tup) 
    return str
bi_df_pos.Bigram= bi_df_pos.Bigram.apply(convertTuple)

In [59]:
bi_df_pos.head()

,Bigram,Count
0,read book,237
1,one best,163
2,highly recommend,133
3,great book,132
4,would recommend,113


In [0]:
bi_df_pos[:200].to_csv('bi_pos.csv', index= False)

## Most Frequent Bigrams in Negative Reviews

In [0]:
tokens = nltk.word_tokenize(' '.join(negative['Clean_reviews']))
#Create your bigrams
bgs = nltk.bigrams(tokens)
key= []
value= []
fdist = nltk.FreqDist(bgs)
for k,v in fdist.items():
    key.append(k)
    value.append(v)
bi_df_neg= pd.DataFrame({'Bigram': key, 'Count': value})
bi_df_neg= bi_df_neg.sort_values(by= 'Count', ascending= False)
bi_df_neg= bi_df_neg.reset_index(drop= True)
def convertTuple(tup): 
    str =  ' '.join(tup) 
    return str
bi_df_neg.Bigram= bi_df_neg.Bigram.apply(convertTuple)

In [62]:
bi_df_neg.head()

,Bigram,Count
0,waste money,233
1,read book,183
2,waste time,173
3,dont waste,166
4,dont buy,136


In [0]:
bi_df_neg[:200].to_csv('bi_neg.csv', index= False)

In [0]:
import spacy
nlp = spacy.load('en_core_web_sm')

## Most Frequent Adjectives in Positive and Reviews

In [0]:
adjectives_column= []
for review in positive.Clean_reviews:
  doc= nlp(review)
  tokens= []
  for token in doc:
    if token.pos_ == 'ADJ':
      tokens.append(str(token))
  adjectives_column.append(" ".join(tokens))
positive['adjective']= adjectives_column

In [0]:
adjectives_neg_column= []
for review in negative.Clean_reviews:
  doc= nlp(review)
  tokens= []
  for token in doc:
    if token.pos_ == 'ADJ':
      tokens.append(str(token))
  adjectives_neg_column.append(" ".join(tokens))
negative['adjective']= adjectives_neg_column

In [67]:
adj_pos_freq = pd.Series(' '.join(positive['adjective']).split()).value_counts()
adj_pos_freq[:10]

great        2061
good         1618
best          658
first         501
little        482
many          474
new           429
excellent     415
easy          330
much          328
dtype: int64

In [68]:
adj_pos_freq[:200].to_csv

<bound method Series.to_csv of great          2061
good           1618
best            658
first           501
little          482
many            474
new             429
excellent       415
easy            330
much            328
old             313
worth           294
wonderful       293
better          292
interesting     238
nice            235
big             232
favorite        229
amazing         228
different       223
classic         217
real            212
perfect         200
beautiful       200
last            192
true            186
long            180
high            164
enough          155
bad             155
               ... 
economic         33
complex          33
natural          33
cute             33
local            33
terrific         32
dead             32
present          32
third            32
average          32
finest           32
easier           32
intense          31
effective        31
surprised        31
catholic         31
honest           31
back     

In [69]:
adj_neg_freq = pd.Series(' '.join(negative['adjective']).split()).value_counts()
adj_neg_freq[:10]

good      1123
bad        737
great      611
first      525
new        478
many       456
much       411
better     396
little     382
worst      364
dtype: int64

In [0]:
adj_neg_freq[:200].to_csv("neg_adj.csv")

# Dashboard

In [71]:
%%html
<div class='tableauPlaceholder' id='viz1566998630104' style='position: relative'><noscript><a href='#'><img alt=' ' src='https:&#47;&#47;public.tableau.com&#47;static&#47;images&#47;Re&#47;ReviewAnalysis_15669976552120&#47;REVIEWANALYTICS&#47;1_rss.png' style='border: none' /></a></noscript><object class='tableauViz'  style='display:none;'><param name='host_url' value='https%3A%2F%2Fpublic.tableau.com%2F' /> <param name='embed_code_version' value='3' /> <param name='path' value='views&#47;ReviewAnalysis_15669976552120&#47;REVIEWANALYTICS?:embed=y&amp;:display_count=y' /> <param name='toolbar' value='yes' /><param name='static_image' value='https:&#47;&#47;public.tableau.com&#47;static&#47;images&#47;Re&#47;ReviewAnalysis_15669976552120&#47;REVIEWANALYTICS&#47;1.png' /> <param name='animate_transition' value='yes' /><param name='display_static_image' value='yes' /><param name='display_spinner' value='yes' /><param name='display_overlay' value='yes' /><param name='display_count' value='yes' /></object></div>                <script type='text/javascript'>                    var divElement = document.getElementById('viz1566998630104');                    var vizElement = divElement.getElementsByTagName('object')[0];                    vizElement.style.width='1138px';vizElement.style.height='2827px';                    var scriptElement = document.createElement('script');                    scriptElement.src = 'https://public.tableau.com/javascripts/api/viz_v1.js';                    vizElement.parentNode.insertBefore(scriptElement, vizElement);                </script>